# MSB-mantissaconversie met vaste schaalverhouding ×64

Eén nieuwe training vanaf nul: vijf epochs met MSB + optimale exponentkeuze. De LSB ×64-cijfers zijn eerder gemeten referentiedata. Deze notebook start geen training.

*Status bij aanmaak van deze notebook; voer de cellen uit om de bestanden opnieuw te lezen.*

## Status: NOG NIET UITGEVOERD

De nieuwe MSB-training is nog niet uitgevoerd of gerapporteerd.

**Nieuwe MSB-epochs afgerond:** 0/5.

De LSB ×64-cijfers zijn **historische metingen**, overgenomen uit de vastgelegde referentierun. In deze uitvoering is geen nieuwe LSB- of RNE-training gedaan.

Dit is training vanaf een verse initialisatie, seed 20261009. Er is geen warmstart, epoch-100-oudermodel of epoch-0-baselinemeting. De vaste schaalverhouding is `S_E2 = 64 × S_E4`; beide schalen kunnen tijdens training veranderen. Er zijn geen geleerde `phi`-parameters.

**Grens van de historische vergelijking:** overeenkomst van historische begin-RNG en samplevolgorde is niet geverifieerd, omdat die historische bestanden ontbreken. Gelijke seed en modelinitialisatie zijn geen bewijs van bitexacte CUDA-uitvoering. Eén nieuwe en één historische run leveren geen onzekerheid over meerdere seeds.

**Gelezen resultatenmap:** `de map met deze notebook en de meetbestanden`


## Resultatenmap

Als de map is verplaatst, pas `RUN_DIR` aan naar de map met `training_report.py`, `run_report.json`, `accuracy.csv` en `reference_lsb.csv`.

In [ ]:
from pathlib import Path
import importlib.util
RUN_DIR = Path.cwd()  # Kies hier zo nodig de volledige resultatenmap.
if not RUN_DIR.is_dir():
    RUN_DIR = Path.cwd()
helper = RUN_DIR / "training_report.py"
if not helper.is_file():
    raise FileNotFoundError("Kies bij RUN_DIR de resultatenmap met training_report.py.")
spec = importlib.util.spec_from_file_location("msb_scale64_report", helper)
tr = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tr)
from IPython.display import Markdown, display


In [ ]:
result = tr.load_results(RUN_DIR)
display(Markdown(tr.summary_markdown(result)))


## Vastgelegde proef

Verse initialisatie met seed 20261009; Adam met begin-LR 0,001 en weight decay 0. De oorspronkelijke milestones 50/75/90 blijven ingesteld en worden binnen vijf epochs niet bereikt. Geen onafhankelijke of bevroren absolute E2-schalen: steeds `S_E2 = 64 × S_E4`. E4M3, E2M3 en E2M1 delen de trainbare FP32-masters binnen de nieuwe MSB-run. De drie BN-banken leren mee; A32, B32 en gezamenlijke recursieve distillatie blijven behouden.

In [ ]:
display(Markdown(tr.endpoint_markdown(result)))


In [ ]:
figures, plot_errors = tr.create_figures(result)
for name, figure in figures.items():
    display(figure)
if figures:
    import matplotlib.pyplot as plt
    for figure in figures.values():
        plt.close(figure)
for message in plot_errors:
    display(Markdown(message))
if not figures and not plot_errors:
    display(Markdown("Nog geen gecontroleerde meetbestanden om te plotten."))


## Interpretatiegrenzen

- De nieuwe MSB-run en de historische LSB-run zijn afzonderlijke uitvoeringen met één seed.
- Hetzelfde beginmodel en protocol bewijzen geen gelijke historische RNG of samplevolgorde.
- Vijf epochs onderzoeken vroege training, niet convergentie of optimale instellingen.
- Door gezamenlijke updates kunnen ook E4M3 en E2M3 tussen de methoden verschillen.
- Het eindpunt is vooraf gekozen op epoch 5; er wordt geen beste testcheckpoint geselecteerd.
- De eerdere post-hocproef gebruikte een andere schaaltoestand en een getraind model. Die accuracies zijn geen baseline voor deze training vanaf nul.
- FP32-emulatie levert geen bewijs van hardwarelatency, energieverbruik of algemene superioriteit.